# 01 · Inspect raw CSIC 2010 and build the feature table

Plan steps 1-3 (see `PLAN.md`): download the three raw files, parse them into requests,
extract features, verify labels and duplicates.

Everything that looks odd here (count mismatches, parse failures, duplicates) goes into
`CONTEXT.md` section 6 and the limitations section.

In [ ]:
# Works both in Google Colab and locally (Jupyter started anywhere inside the repo).
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/<your-username>/anomaly-detection-paper.git"  # TODO: set after pushing to GitHub

if IN_COLAB:
    if not Path("anomaly-detection-paper").exists():
        !git clone -q {REPO_URL}
    os.chdir("anomaly-detection-paper")
else:
    while not Path("CONTEXT.md").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("Repo root:", ROOT)

## 1. Download the raw files (skips files that already exist)

In [ ]:
!python src/download.py

## 2. Look at a few raw requests before trusting the parser

In [ ]:
raw = (ROOT / "data/raw/anomalousTrafficTest.txt").read_text(encoding="latin-1")
print(raw[:3000])

## 3. Parse, extract features, write CSVs and the data report

In [ ]:
!python src/build_dataset.py

In [ ]:
import json
report = json.loads((ROOT / "results/metrics/data_report.json").read_text(encoding="utf-8"))
for f, info in report["files"].items():
    print(f"{f:28s} parsed={info['parsed_requests']:>6}  expected={info['expected']}  "
          f"failures={info['parse_failures']}  dup_within={info['duplicates_within_file']}  methods={info['methods']}")
print()
print(json.dumps(report["duplicates"], indent=2, ensure_ascii=False)[:2000])

**Record here (and in CONTEXT.md §6):**
- Parsed counts vs official counts:
- Parse failures:
- Duplicates between normal and anomalous test sets (label noise → limitations):

## 4. Sanity-check the feature table

In [ ]:
import pandas as pd
from features import FEATURE_NAMES

df = pd.read_csv(ROOT / "data/processed/csic2010_features.csv")
print(df.groupby(["split", "label"]).size())
df[FEATURE_NAMES].describe().T

In [ ]:
# Mean of each feature for normal vs anomalous test requests (descriptive only, not a result).
test = df[df.split == "test"]
test.groupby("label")[FEATURE_NAMES].mean().T.rename(columns={0: "normal", 1: "anomalous"})

In [ ]:
import matplotlib.pyplot as plt

cols = ["query_length", "special_char_count", "pct_escape_count", "payload_entropy",
        "max_param_value_length", "non_alnum_ratio"]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, c in zip(axes.flat, cols):
    for lab, name in [(0, "normal"), (1, "anomalous")]:
        ax.hist(test.loc[test.label == lab, c], bins=50, alpha=0.6, label=name, density=True)
    ax.set_title(c)
axes[0, 0].legend()
fig.tight_layout()
fig.savefig(ROOT / "results/figures/feature_distributions.png", dpi=150)